In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

# ==========================
# CONFIGURATION
# ==========================
source_dir = "VoiceDS_SHR"          # Path to original dataset
output_dir = "dataset_split"    # Output folder

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

random_seed = 42

# ==========================
# CREATE OUTPUT FOLDERS
# ==========================
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# ==========================
# SPLIT EACH CLASS
# ==========================
classes = [
    d for d in os.listdir(source_dir)
    if os.path.isdir(os.path.join(source_dir, d))
]

for cls in classes:

    class_path = os.path.join(source_dir, cls)

    images = [
        f for f in os.listdir(class_path)
        if os.path.isfile(os.path.join(class_path, f))
    ]

    # Train split
    train_files, temp_files = train_test_split(
        images,
        test_size=(1 - train_ratio),
        random_state=random_seed,
        shuffle=True
    )

    # Validation and Test split
    val_files, test_files = train_test_split(
        temp_files,
        test_size=test_ratio / (val_ratio + test_ratio),
        random_state=random_seed,
        shuffle=True
    )

    # Create class folders
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

    # Copy files
    for f in train_files:
        shutil.copy2(
            os.path.join(class_path, f),
            os.path.join(output_dir, "train", cls, f)
        )

    for f in val_files:
        shutil.copy2(
            os.path.join(class_path, f),
            os.path.join(output_dir, "val", cls, f)
        )

    for f in test_files:
        shutil.copy2(
            os.path.join(class_path, f),
            os.path.join(output_dir, "test", cls, f)
        )

    print(f"{cls}:")
    print(f"  Train: {len(train_files)}")
    print(f"  Val:   {len(val_files)}")
    print(f"  Test:  {len(test_files)}")

print("\nDataset successfully split!")

dysar:
  Train: 134
  Val:   29
  Test:  29
Laryngitis:
  Train: 293
  Val:   63
  Test:  64
parkinson:
  Train: 305
  Val:   66
  Test:  66
Vox senilis:
  Train: 274
  Val:   59
  Test:  59

Dataset successfully split!


In [2]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# ==========================
# Paths
# ==========================
INPUT_DIR = Path("dataset_split")
OUTPUT_DIR = Path("spectrogram_dataset")

# ==========================
# Parameters
# ==========================
SR = 16000
N_FFT = 1024
HOP_LENGTH = 160      # 10 ms
WIN_LENGTH = 400      # 25 ms
N_MELS = 224
TARGET_SIZE = (224, 224)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

wav_files = list(INPUT_DIR.rglob("*.wav"))

print(f"Found {len(wav_files)} wav files.")

for wav_path in wav_files:

    # Preserve directory structure
    rel = wav_path.relative_to(INPUT_DIR)
    save_path = OUTPUT_DIR / rel.with_suffix(".png")
    save_path.parent.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Load audio
    # -----------------------------
    y, sr = librosa.load(
        wav_path,
        sr=SR,
        mono=True
    )

    # Normalize
    y = librosa.util.normalize(y)

    # -----------------------------
    # Mel Spectrogram
    # -----------------------------
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        power=2.0,
        fmin=20,
        fmax=sr // 2,
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80
    )

    # -----------------------------
    # Normalize to 0-255
    # -----------------------------
    mel_db -= mel_db.min()
    mel_db /= mel_db.max() + 1e-8
    mel_img = (mel_db * 255).astype(np.uint8)

    # Convert to RGB
    img = Image.fromarray(mel_img)
    img = img.resize(TARGET_SIZE, Image.Resampling.BICUBIC)
    img = img.convert("RGB")

    img.save(save_path)

print("Done.")

Found 1441 wav files.
Done.
